# Continuous Batching：请求级随机流保证批次不变采样

**面试问题：请求加入或退出动态 Batch 时，为什么同一 seed 仍可能生成不同文本，怎样修复？**

## 回答主线

1. Continuous Batching 每个 Decode tick 都会重组活跃请求，吞吐高于等待整批结束的静态批处理。
2. 如果所有请求共享一个全局 RNG，随机数消费顺序会随 Batch 成员变化，同一请求便无法复现。
3. 请求级 RNG 仍可能受重试和调度影响，更稳妥的是用 request seed、Token 位置和采样流编号生成 counter-based 随机数。
4. 每个 Token 的概率分布不变时，无论请求单独运行、交错运行或恢复运行，都应得到同一个采样值。
5. request_id 或 nonce 复用会让不同用户产生相关随机流，因此随机键也属于请求合同。
6. 评估要同时检查批次不变性、调度 tick、活跃槽利用率和延迟。

## 真实案例

六个客服生成请求在不同 tick 到达，输出长度为 2–5 Token，共享一个五词词表。我们先展示全局 RNG 在“单独执行”和“交错执行”下改变 R1 输出，再用 SHA-256 实现位置计数随机流，并运行一个动态加入/退出的连续批调度器。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：六个到达时间和概率分布

In [1]:
import hashlib  # 导入哈希函数以实现确定性的计数随机流。
import random  # 导入伪随机数以复现共享 RNG 的错误基线。

vocabulary = ["已", "为您", "查询", "处理", "完成"]  # 定义可读客服输出词表。
probabilities = [0.18, 0.22, 0.20, 0.24, 0.16]  # 定义每个位置共享的教学采样分布。
requests = [  # 构造六个具有到达时间和长度的请求。
    {"id": "R1", "arrival": 0, "length": 5, "seed": 101, "prompt": "查询退款"},  # 长回复请求。
    {"id": "R2", "arrival": 0, "length": 2, "seed": 202, "prompt": "确认地址"},  # 同 tick 到达的短请求。
    {"id": "R3", "arrival": 1, "length": 4, "seed": 303, "prompt": "查看物流"},  # 第二 tick 动态加入。
    {"id": "R4", "arrival": 2, "length": 3, "seed": 404, "prompt": "修改发票"},  # 第三 tick 动态加入。
    {"id": "R5", "arrival": 3, "length": 2, "seed": 505, "prompt": "取消订单"},  # 后到短请求。
    {"id": "R6", "arrival": 4, "length": 3, "seed": 606, "prompt": "核验账号"},  # 最后加入的请求。
]  # 完成调度输入。
print("请求  arrival  length  seed  prompt")  # 输出输入表头。
for request in requests:  # 逐请求展示调度字段。
    print(f"{request['id']} {request['arrival']:>8} {request['length']:>7} {request['seed']:>5}  {request['prompt']}")  # 展示长短请求混合。

请求  arrival  length  seed  prompt
R1        0       5   101  查询退款
R2        0       2   202  确认地址
R3        1       4   303  查看物流
R4        2       3   404  修改发票
R5        3       2   505  取消订单
R6        4       3   606  核验账号


## Baseline 基线：共享全局 RNG 的输出依赖执行顺序

In [2]:
def sample_from_u(random_value):  # 根据一个 [0,1) 随机数从离散分布采样。
    cumulative = 0.0  # 初始化累计概率。
    for token, probability in zip(vocabulary, probabilities):  # 按词表顺序扫描概率区间。
        cumulative += probability  # 扩展当前 Token 的累计上界。
        if random_value < cumulative:  # 随机数落入当前区间。
            return token  # 返回采样 Token。
    return vocabulary[-1]  # 处理浮点边界并返回最后 Token。

def shared_rng_run(order, seed):  # 按给定请求交错顺序消费一个全局 RNG。
    generator = random.Random(seed)  # 创建所有请求共享的随机流。
    outputs = {request_id: [] for request_id in set(order)}  # 初始化每个请求的输出。
    for request_id in order:  # 调度器每安排一步就消费下一个随机数。
        outputs[request_id].append(sample_from_u(generator.random()))  # 把采样 Token 追加到当前请求。
    return outputs  # 返回按请求聚合的结果。

solo_order = ["R1"] * 5  # 构造 R1 单独执行的随机数消费顺序。
interleaved_order = ["R1", "R2", "R1", "R2", "R1", "R3", "R1", "R3", "R1"]  # 构造 R1 与短请求交错的顺序。
solo_shared = shared_rng_run(solo_order, seed=77)["R1"]  # 运行单独请求基线。
batch_shared = shared_rng_run(interleaved_order, seed=77)["R1"]  # 运行交错 Batch 基线。
print("共享 RNG，R1 单独执行：", solo_shared)  # 展示第一种 Token 序列。
print("共享 RNG，R1 交错执行：", batch_shared)  # 展示 Batch 成员改变后的序列。
print("同一 seed 是否复现：", solo_shared == batch_shared)  # 直接暴露批次依赖。

共享 RNG，R1 单独执行： ['处理', '为您', '为您', '处理', '已']
共享 RNG，R1 交错执行： ['处理', '为您', '已', '查询', '已']
同一 seed 是否复现： False


### 核心实现：request seed × Token 位置的计数随机数

In [3]:
def counter_uniform(request_seed, token_index, stream=0):  # 从请求 seed、位置和流编号生成确定性随机数。
    key = f"{request_seed}:{token_index}:{stream}".encode("utf-8")  # 形成与调度顺序无关的计数键。
    digest = hashlib.sha256(key).digest()  # 计算稳定的 256 位摘要。
    integer = int.from_bytes(digest[:8], "big")  # 读取前八字节形成 64 位整数。
    return integer / 2**64  # 映射到 [0,1) 区间。

def sample_request(request, positions):  # 对指定 Token 位置进行请求级采样。
    rows = []  # 收集位置、随机值和 Token。
    for position in positions:  # 遍历调度器实际安排的位置。
        random_value = counter_uniform(request["seed"], position)  # 用绝对位置读取随机数而非顺序消费。
        rows.append({"position": position, "u": random_value, "token": sample_from_u(random_value)})  # 保存可回放采样记录。
    return rows  # 返回位置级采样结果。

r1 = requests[0]  # 选择 R1 演示请求级随机流。
r1_solo_rows = sample_request(r1, list(range(r1["length"])))  # 按连续位置单独采样。
r1_split_rows = sample_request(r1, [0, 2, 4]) + sample_request(r1, [1, 3])  # 模拟位置被不同 Batch tick 交错执行。
r1_split_rows = sorted(r1_split_rows, key=lambda row: row["position"])  # 按绝对 Token 位置恢复输出顺序。
print("position  uniform              token")  # 输出计数采样中间量表头。
for row in r1_solo_rows:  # 逐位置展示随机数与 Token。
    print(f"{row['position']:>3}       {row['u']:.12f}  {row['token']}")  # 展示每个位置的稳定值。
print("拆分调度后 R1：", [row["token"] for row in r1_split_rows])  # 展示交错执行不改变位置结果。

position  uniform              token
  0       0.487780977715  查询
  1       0.133233335909  已
  2       0.105846104782  已
  3       0.749944395730  处理
  4       0.334271161869  为您
拆分调度后 R1： ['查询', '已', '已', '处理', '为您']


## 结果解读：运行真正的动态加入/退出调度器

In [4]:
def continuous_batch(items):  # 模拟每个 tick 为所有活跃请求生成一个 Token。
    active = []  # 保存当前活跃请求状态。
    outputs = {item["id"]: [] for item in items}  # 初始化每个请求输出。
    timeline = []  # 保存每个 tick 的 Batch 成员。
    tick = 0  # 从第零个调度 tick 开始。
    while any(len(outputs[item["id"]]) < item["length"] for item in items):  # 直到全部请求达到目标长度。
        for item in items:  # 扫描当前 tick 新到达的请求。
            if item["arrival"] == tick:  # 当前请求恰好到达。
                active.append(item)  # 把请求加入动态 Batch。
        member_ids = [item["id"] for item in active]  # 记录本 tick 执行前的活跃成员。
        for item in active:  # 每个活跃请求生成一个位置的 Token。
            position = len(outputs[item["id"]])  # 绝对位置等于当前已生成长度。
            random_value = counter_uniform(item["seed"], position)  # 获取与 Batch 无关的随机值。
            outputs[item["id"]].append(sample_from_u(random_value))  # 提交当前 Token。
        active = [item for item in active if len(outputs[item["id"]]) < item["length"]]  # 移除本 tick 已完成的请求。
        timeline.append({"tick": tick, "members": member_ids, "remaining": [item["id"] for item in active]})  # 保存调度轨迹。
        tick += 1  # 推进到下一个 Decode tick。
    return outputs, timeline  # 返回所有输出和 Batch 时间线。

batch_outputs, timeline = continuous_batch(requests)  # 执行六个请求的动态批处理。
print("tick  执行成员                 tick后仍活跃")  # 输出调度时间线表头。
for event in timeline:  # 逐 tick 展示加入和退出。
    print(f"{event['tick']:>2}    {event['members']} -> {event['remaining']}")  # 展示 Batch 形状变化。
print("请求  单独输出                  动态Batch输出              一致")  # 输出批次不变性结果表头。
for request in requests:  # 将每个请求单独采样结果与动态结果对照。
    solo = [row["token"] for row in sample_request(request, list(range(request["length"])))]  # 生成单独运行参照。
    print(f"{request['id']}   {str(solo):<27} {str(batch_outputs[request['id']]):<27} {solo == batch_outputs[request['id']]}")  # 展示六条均一致。
print("解读：活跃集合每 tick 改变，但随机键只取决于请求和绝对位置，所以调度不进入采样语义。")  # 解释不变量。

tick  执行成员                 tick后仍活跃
 0    ['R1', 'R2'] -> ['R1', 'R2']
 1    ['R1', 'R2', 'R3'] -> ['R1', 'R3']
 2    ['R1', 'R3', 'R4'] -> ['R1', 'R3', 'R4']
 3    ['R1', 'R3', 'R4', 'R5'] -> ['R1', 'R3', 'R4', 'R5']
 4    ['R1', 'R3', 'R4', 'R5', 'R6'] -> ['R6']
 5    ['R6'] -> ['R6']
 6    ['R6'] -> []
请求  单独输出                  动态Batch输出              一致
R1   ['查询', '已', '已', '处理', '为您'] ['查询', '已', '已', '处理', '为您'] True
R2   ['查询', '为您']                ['查询', '为您']                True
R3   ['完成', '查询', '完成', '完成']    ['完成', '查询', '完成', '完成']    True
R4   ['查询', '处理', '处理']          ['查询', '处理', '处理']          True
R5   ['已', '查询']                 ['已', '查询']                 True
R6   ['为您', '处理', '为您']          ['为您', '处理', '为您']          True
解读：活跃集合每 tick 改变，但随机键只取决于请求和绝对位置，所以调度不进入采样语义。


## 失败案例：复用 seed 与 request nonce 产生相关回复

In [5]:
duplicate_a = {"id": "retry-A", "seed": 909, "length": 4}  # 构造第一个请求随机身份。
duplicate_b = {"id": "retry-B", "seed": 909, "length": 4}  # 错误地复用同一 seed 作为第二个用户请求。
tokens_a = [row["token"] for row in sample_request(duplicate_a, list(range(4)))]  # 采样第一个请求。
tokens_b = [row["token"] for row in sample_request(duplicate_b, list(range(4)))]  # 采样第二个请求。
unique_b = {"id": "retry-B", "seed": 910, "length": 4}  # 为第二个逻辑请求分配唯一 nonce 派生 seed。
tokens_unique_b = [row["token"] for row in sample_request(unique_b, list(range(4)))]  # 使用独立随机流重采样。
print(f"复用 seed：A={tokens_a} B={tokens_b} 完全相同={tokens_a == tokens_b}")  # 展示跨用户随机流相关。
print(f"唯一 nonce：A={tokens_a} B={tokens_unique_b} 完全相同={tokens_a == tokens_unique_b}")  # 展示修正后解耦。
print("修正策略：随机键使用不可复用的 generation_id + user_seed + token_index；重试是否复现由显式 retry_of 决定。")  # 总结请求身份合同。

复用 seed：A=['完成', '已', '处理', '为您'] B=['完成', '已', '处理', '为您'] 完全相同=True
唯一 nonce：A=['完成', '已', '处理', '为您'] B=['处理', '为您', '查询', '完成'] 完全相同=False
修正策略：随机键使用不可复用的 generation_id + user_seed + token_index；重试是否复现由显式 retry_of 决定。


### 生产边界与采样事件

In [6]:
sampling_event = {"request_id": "R1", "generation_id": "g-101-1", "seed": r1["seed"], "token_index": 2, "stream": 0, "u": round(r1_solo_rows[2]["u"], 12), "sampler_version": "counter-r1"}  # 构造单 Token 可回放事件。
print("采样事件：", sampling_event)  # 展示复现需要的最小字段。
print("生产替换点：真实 Serving 还需 GPU counter RNG、Top-k/Top-p、浮点确定性、Beam 分支流、取消恢复、跨副本 generation_id 和吞吐基准。")  # 明确 Python 哈希教学边界。

采样事件： {'request_id': 'R1', 'generation_id': 'g-101-1', 'seed': 101, 'token_index': 2, 'stream': 0, 'u': 0.105846104782, 'sampler_version': 'counter-r1'}
生产替换点：真实 Serving 还需 GPU counter RNG、Top-k/Top-p、浮点确定性、Beam 分支流、取消恢复、跨副本 generation_id 和吞吐基准。


## 回归测试：最后只保护批次不变性与随机身份

In [7]:
assert solo_shared != batch_shared  # 验证共享 RNG 的批次依赖失败探针稳定存在。
assert [row["token"] for row in r1_solo_rows] == [row["token"] for row in r1_split_rows]  # 验证位置打乱不改变 R1 序列。
assert all(batch_outputs[request["id"]] == [row["token"] for row in sample_request(request, list(range(request["length"])))] for request in requests)  # 验证六个请求动态与单独执行一致。
assert any(len(event["members"]) != len(timeline[0]["members"]) for event in timeline[1:])  # 验证测试确实经历动态 Batch 形状变化。
assert tokens_a == tokens_b and tokens_a != tokens_unique_b  # 验证 seed 复用相关性与唯一 nonce 修正。
print("回归测试通过：共享 RNG 反例、位置计数、六请求批次不变性、动态形状和随机身份均成立。")  # 用少量断言总结采样合同。

回归测试通过：共享 RNG 反例、位置计数、六请求批次不变性、动态形状和随机身份均成立。
